### Marketing Mix Model (MMM)

MMM weekly data of 2 years translate to 104 weeks of data. This makes MMM a small data problem, and carefully handling our adjusted explained variance metrics is very important.

R-Squared: Explained variance

Adjusted R-Squared: Adjusted Explained variance

As data scientist or statistician or some applying statistical or machine learning models we always expect our Adjusted R-squared to penalize our error more than R-squared. That is we always expect Adjsuted R-squared to have equal or less good performance value than R-Squared.

That is When:

R-Sqaured = 90%

Adjusted R-Squared: 88%


#### Formular

$$R^2 = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$$
$$R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}}$$

$$R^2_{\text{adj}} = 1 - \left[ \frac{(1 - R^2)(n - 1)}{n - p - 1} \right]$$

Both $R^2
and
Adjusted
  $R^2 ranges from 0 to 1, 0 poorly fit
  and 1 perfectly fit


#### MMM ideal data example

n: 104

p: 10

n: Number of Samples/Observations in the data

p: Number of Predictor/Features/Variables in the data

#### Out of Sample method

The OOS method used is Time Holdout, for example we fit our training data on 99 weeks (95%) of the data and test the performance on 5 data point (5%), note the spliting must be in chronological order, i.e week 100 is not before week 99 unlike random train test split that time order is not taken into place

In [1]:
import numpy as np
import pandas as pd

In [2]:
def get_rsq(y_true, y_pred, p=None, df_int=1, n=None):

    """Calculates Adjusted R2 for an MMM model.

    Parameters:
    y_true (array-like): Actual target values (e.g., Sales/Conversions)
    y_pred (array-like): Model predicted values
    p (int): Number of independent variables (media + non-media + organic)
    n: Number of observation

    Returns:
    float: R2 score
    float: Adjusted R2 score
    """

    rss = np.sum((y_true - y_pred) ** 2) # sum of squared residual
    tss = np.sum((y_true - np.mean(y_true)) ** 2) # sum of squared total

    r2 = 1 - rss / tss

    if p is None:
        return r2

    n = n if n is not None else len(y_true)

    rdf = n - p - 1
    r2_adj = 1 - (1 - r2) * ((n - df_int) / rdf)

    return r2, r2_adj

In [3]:
def get_adj_r2(r2, p=None, df_int=1, n=None):

    """Calculates Adjusted R2 for an MMM model.

    Parameters:
    r2 (float): R2 score
    p (int): Number of predictors
    n: Number of observation

    Returns:
    float: R2 score
    float: Adjusted R2 score
    """
    r2 = r2

    if p is None:
        return r2

    rdf = n - p - 1
    r2_adj = 1 - (1 - r2) * ((n - df_int) / rdf)

    return {'Fixed_R-squared':r2, 'Adjusted_R-squared':round(r2_adj,3)}

### Example 1

N: 104 (total number of data point)

training data: 99 rows (95%)

validataion data: 5 rows (5%)

Fixed training R-squared (In-sample): -0.2

Fixed validation R-squared (In-sample): -0.3

In [4]:
## Training evaluation
get_adj_r2(-0.2, p=10, n=99)

{'Fixed_R-squared': -0.2, 'Adjusted_R-squared': -0.336}

In [5]:
### Test evaluation
get_adj_r2(-0.3, p=10, n=5)

{'Fixed_R-squared': -0.3, 'Adjusted_R-squared': 1.867}

#### Example 1 Conclusion

We can see that our settings did not have problem with the training evaluation but a problem in validation evaluation, why?

Training evaluation: n > p+1

Validation evaluation: n < p

### Example 2

N: 104 (total number of data point)

training data: 99 rows (95%)

validataion data: 5 rows (5%)

Fixed training R-squared (In-sample): -0.2

Fixed validation R-squared (In-sample): -0.3

In [6]:
## Training evaluation
get_adj_r2(-0.2, p=10, n=99)

{'Fixed_R-squared': -0.2, 'Adjusted_R-squared': -0.336}

In [7]:
### Test evaluation
get_adj_r2(-0.3, p=10, n=99)

{'Fixed_R-squared': -0.3, 'Adjusted_R-squared': -0.448}

In [8]:
## Futher test with positive R-squared
### Test evaluation
get_adj_r2(0.9, p=10, n=5)

{'Fixed_R-squared': 0.9, 'Adjusted_R-squared': 1.067}

#### Example 2 Conclusion

We can see that our settings did not have problem with the training evaluation and validation evaluation, why?

Training evaluation: n > p+1

Validation evaluation: n > p+1

So, ideally since MMM is small data problem, our plug in number of observation to calculate our out of sample evaluation must include the number of observations in our training data, to satisfy the assumption of n>p+1

### Guardrails way

In [9]:
def get_adj_r2_guardrail(r2, p=None, df_int=1, n=None):

    """Calculates Adjusted R2 for an MMM model.

    Parameters:
    r2 (float): R2 score
    p (int): Number of predictors
    n: Number of observation

    Returns:
    float: R2 score
    float: Adjusted R2 score
    """
    r2 = r2

    if p is None:
        return r2

    rdf = n - p - 1

    if rdf <= 0:
        raise ValueError(
            f"Degrees of freedom error: Sample size n ({n}) must be greater than "
            f"the number of features p ({p}) plus 1."
        )

    r2_adj = 1 - (1 - r2) * ((n - df_int) / rdf)

    return {'Fixed_R-squared':r2, 'Adjusted_R-squared':round(r2_adj,3)}

In [10]:
get_adj_r2_guardrail(-0.3, p=10, n=5)

ValueError: Degrees of freedom error: Sample size n (5) must be greater than the number of features p (10) plus 1.

In [11]:
get_adj_r2_guardrail(-0.3, p=10, n=12)

{'Fixed_R-squared': -0.3, 'Adjusted_R-squared': -13.3}

In [12]:
get_adj_r2_guardrail(-0.3, p=10, n=99)

{'Fixed_R-squared': -0.3, 'Adjusted_R-squared': -0.448}

### Conclusion

The guardrails produce an error since n < p+1, but it produce a result when n > p+1 but we can see a large penalization from Adjusted R-Squared (n=12), but when we use the training observation (n=99), the result resemble much more we expected.

### MMM ideal method

In [13]:
def get_rsq(y_true, y_pred, p=None, df_int=1, n=None):

    """Calculates Adjusted R2 for an MMM model.

    Parameters:
    y_true (array-like): Actual target values (e.g., Sales/Conversions)
    y_pred (array-like): Model predicted values
    p (int): Number of independent variables (media + non-media + organic)
    n: Number of observation

    Returns:
    float: R2 score
    float: Adjusted R2 score
    """

    rss = np.sum((y_true - y_pred) ** 2) # sum of squared residual
    tss = np.sum((y_true - np.mean(y_true)) ** 2) # sum of squared total

    r2 = 1 - rss / tss

    if p is None:
        return r2

    n = n if n is not None else len(y_true) ## if our n is not provided use the out of sample length

    rdf = n - p - 1
    r2_adj = 1 - (1 - r2) * ((n - df_int) / rdf)

    return r2, r2_adj

In [14]:
## AI generated

np.random.seed(42)

# 104 weeks
n = 104

weeks = pd.date_range("2024-01-01", periods=n, freq="W")

# Trend
trend = np.arange(n)

# Holidays (about 10% of weeks)
holiday = np.random.binomial(1, 0.10, n)

# Promotion (about 20% of weeks)
promotion = np.random.binomial(1, 0.20, n)

# Price index
price_index = np.random.normal(100, 3, n)

# Distribution
distribution = np.clip(
    70 + np.linspace(0, 15, n) + np.random.normal(0, 2, n),
    60,
    95
)

# Media spends
tv = np.random.gamma(8, 1200, n)
meta = np.random.gamma(10, 400, n)
google = np.random.gamma(12, 250, n)
youtube = np.random.gamma(6, 500, n)
radio = np.random.gamma(5, 300, n)

# Sales generation
sales = (
    10000
    + 0.05 * tv
    + 0.10 * meta
    + 0.15 * google
    + 0.08 * youtube
    + 0.06 * radio
    - 40 * (price_index - 100)
    + 300 * promotion
    + 45 * distribution
    + 500 * holiday
    + 12 * trend
    + np.random.normal(0, 100, n)
)

df = pd.DataFrame({
    "week": weeks,
    "sales": sales,
    "tv": tv,
    "meta": meta,
    "google": google,
    "youtube": youtube,
    "radio": radio,
    "price_index": price_index,
    "promotion": promotion,
    "distribution": distribution,
    "holiday": holiday,
    "trend": trend
})

In [15]:
df.shape

(104, 12)

In [16]:
train = df[:99]
val = df[99:]

In [17]:
X_train = train.drop(['sales', 'week'], axis=1)
y_train = train['sales']

X_val = val.drop(['sales', 'week'], axis=1)
y_val = val['sales']

In [18]:
import statsmodels.api as sm

In [19]:
X_train = sm.add_constant(X_train)

In [20]:
model = sm.OLS(y_train, X_train).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.976
Model:                            OLS   Adj. R-squared:                  0.973
Method:                 Least Squares   F-statistic:                     359.9
Date:                Tue, 07 Jul 2026   Prob (F-statistic):           7.40e-67
Time:                        10:18:26   Log-Likelihood:                -589.63
No. Observations:                  99   AIC:                             1201.
Df Residuals:                      88   BIC:                             1230.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const         1.453e+04    521.713     27.852   

**Note**

The R-squared and Adjusted R-squared, 0.687 and 0.652 respectively is for the In-sample fit, we can test on our own too with our **get_rsq** to see if our maths are right. Then we see that Adjusted $R^2$ penalize the performance of the model more, 0.976 < 0.973

In [21]:
### Our In-Sample prediction
y_pred_train = model.predict(X_train)

In [22]:
get_rsq(y_train, y_pred_train, p=10, df_int=1) ## In-sample

(np.float64(0.9761353930446518), np.float64(0.973423505890635))

In [23]:
### Our Out-Sample prediction
X_val = sm.add_constant(X_val)
y_pred = model.predict(X_val)

In [24]:
### Now evaluation on the validation data
get_rsq(y_val, y_pred, p=10, df_int=1) ## Out-Of-Sample Fit

(np.float64(0.12217948376579768), np.float64(1.585213677489468))

**Note**

We observe inflated Adjusted $R^2$ which is not the reality of the metrics

In [25]:
### Now evaluation on the validation data
get_rsq(y_val, y_pred, p=10, n=len(X_train)) ## Out-Of-Sample Fit let provide our training n

(np.float64(0.12217948376579768), np.float64(0.022427152375547488))

**Note**

Our expected reality is achieved